# BraTS 2021 Preprocessing Pipeline — All Fixes Applied
**SEIS 766 Vision AI — Spring 2026**

Corrected task ordering: `SEG-01 → SEG-02 → SEG-03 → SEG-06 → SEG-04 → SEG-05`

All fixes from code review applied:
1. SEG-01: Spot-check 3 random cases with shape verification
2. SEG-03: `np.nan_to_num()` before normalization
3. SEG-06: `.copy()` before shuffle to protect caller's list
4. SEG-06: Check if split JSON already exists
5. SEG-04: `+1` on end_z to include full margin
6. SEG-04: Partition-aware extraction (train/val saved separately)
7. SEG-05: `.copy()` after `np.flip` for array contiguity
8. SEG-05: Geometric and photometric transforms separated
9. SEG-05: `borderMode`/`borderValue` in warpAffine
10. SEG-05: Mask cast to float32 before warp, uint8 after
11. SEG-05: Wrapped in Dataset class (train augments, val doesn't)
12. Execution: leakage verification check


In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/"


Mounted at /content/drive


In [2]:
import os
import json
import random
import numpy as np
import cv2
import nibabel as nib

# ═══════════════════════════════════════════════════════════════
# PATHS — The BraTS case folders (BraTS2021_00000, etc.) live
# directly inside Group_Project_766, not in a subfolder.
# ═══════════════════════════════════════════════════════════════
TRAIN_PATH = ROOT + "Group_Project_766"
OUTPUT_DIR = ROOT + "Group_Project_766"

print(f"TRAIN_PATH exists: {os.path.exists(TRAIN_PATH)}")
if os.path.exists(TRAIN_PATH):
    contents = os.listdir(TRAIN_PATH)[:5]
    print(f"First 5 items: {contents}")


TRAIN_PATH exists: True
First 5 items: ['BraTS2021_01442', 'BraTS2021_01449', 'BraTS2021_01439', 'BraTS2021_01443', 'BraTS2021_01450']


In [3]:
# ═══════════════════════════════════════════════════════════════
# SEG-01: Verify BraTS 2021 Dataset
# Spot-checks 3 random cases for file presence + shape (240x240x155)
# ═══════════════════════════════════════════════════════════════

def verify_data():
    if not os.path.exists(TRAIN_PATH):
        print(f"❌ Folder not found at: {TRAIN_PATH}")
        return []

    # Find all BraTS case FOLDERS (skip any loose files)
    cases = sorted([d for d in os.listdir(TRAIN_PATH)
                    if d.startswith("BraTS2021") and
                    os.path.isdir(os.path.join(TRAIN_PATH, d))])
    print(f"📂 Found {len(cases)} training cases")

    if not cases:
        print("❌ No BraTS2021 case folders found.")
        return []

    # Spot-check 3 random cases for integrity
    spot_checks = random.sample(cases, min(3, len(cases)))
    expected_suffixes = ["_t1.nii.gz", "_t1ce.nii.gz",
                         "_t2.nii.gz", "_flair.nii.gz", "_seg.nii.gz"]

    for case_id in spot_checks:
        case_path = os.path.join(TRAIN_PATH, case_id)
        for suffix in expected_suffixes:
            fpath = os.path.join(case_path, f"{case_id}{suffix}")
            if not os.path.exists(fpath):
                print(f"  ⚠️  Missing: {fpath}")
                continue
            img = nib.load(fpath)
            if img.shape != (240, 240, 155):
                print(f"  ⚠️  Unexpected shape {img.shape} in {fpath}")
            else:
                print(f"  ✅ {case_id}{suffix} → {img.shape}")

    return cases

all_cases = verify_data()


📂 Found 1251 training cases
  ✅ BraTS2021_01080_t1.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_t1ce.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_t2.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_flair.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01080_seg.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_t1.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_t1ce.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_t2.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_flair.nii.gz → (240, 240, 155)
  ✅ BraTS2021_00563_seg.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_t1.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_t1ce.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_t2.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_flair.nii.gz → (240, 240, 155)
  ✅ BraTS2021_01139_seg.nii.gz → (240, 240, 155)


In [4]:
# ═══════════════════════════════════════════════════════════════
# SEG-02: NIfTI Loading Pipeline
# ═══════════════════════════════════════════════════════════════

def load_case_data(case_id):
    """Load all 4 modalities + seg mask for one case."""
    case_path = os.path.join(TRAIN_PATH, case_id)
    modalities = ['t1', 't1ce', 't2', 'flair']
    data_dict = {}

    for mod in modalities:
        file_path = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
        img = nib.load(file_path)
        data_dict[mod] = img.get_fdata().astype(np.float32)

    seg_path = os.path.join(case_path, f"{case_id}_seg.nii.gz")
    seg_img = nib.load(seg_path)
    data_dict['seg'] = seg_img.get_fdata().astype(np.uint8)

    return data_dict

# Quick test
if all_cases:
    sample = load_case_data(all_cases[0])
    print(f"✅ Loaded {all_cases[0]}:")
    for key, value in sample.items():
        print(f"   {key}: {value.shape} {value.dtype}")


✅ Loaded BraTS2021_00000:
   t1: (240, 240, 155) float32
   t1ce: (240, 240, 155) float32
   t2: (240, 240, 155) float32
   flair: (240, 240, 155) float32
   seg: (240, 240, 155) uint8


In [5]:
# ═══════════════════════════════════════════════════════════════
# SEG-03: Per-Volume Z-Score Normalization
#
# FIX: np.nan_to_num() added as first line to catch NaN/Inf
# from scanner artifacts before they corrupt mean/std.
# ═══════════════════════════════════════════════════════════════

def normalize_volume(volume):
    # FIX: catch scanner artifacts
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)

    brain_mask = volume > 0
    if np.any(brain_mask):
        mean = volume[brain_mask].mean()
        std = volume[brain_mask].std()
        normalized = (volume - mean) / (std + 1e-8)
        normalized[~brain_mask] = 0
        return normalized.astype(np.float32)
    return volume.astype(np.float32)


def normalize_case(data_dict):
    """Normalize all 4 modalities. Seg mask is untouched."""
    for mod in ['t1', 't1ce', 't2', 'flair']:
        data_dict[mod] = normalize_volume(data_dict[mod])
    return data_dict

# Quick test
if all_cases:
    norm_flair = normalize_volume(sample['flair'])
    tissue = norm_flair[norm_flair != 0]
    print(f"📊 Normalization check ({all_cases[0]} FLAIR):")
    print(f"   Original max: {sample['flair'].max():.2f}")
    print(f"   Normalized mean: {tissue.mean():.4f}")
    print(f"   Normalized std:  {tissue.std():.4f}")


📊 Normalization check (BraTS2021_00000 FLAIR):
   Original max: 2934.00
   Normalized mean: -0.0000
   Normalized std:  1.0000


In [6]:
# ═══════════════════════════════════════════════════════════════
# SEG-06: Case-Level Train/Val Split
#
# MUST run BEFORE SEG-04 (slice extraction) to prevent leakage.
# FIX: .copy() before shuffle so caller's list is not modified.
# FIX: Check if split JSON already exists before recreating.
# ═══════════════════════════════════════════════════════════════

def save_dataset_split(case_ids, train_ratio=0.8):
    random.seed(42)
    # FIX: .copy() so original list is not permanently shuffled
    shuffled = case_ids.copy()
    random.shuffle(shuffled)

    split_point = int(len(shuffled) * train_ratio)
    train_list = shuffled[:split_point]
    val_list = shuffled[split_point:]

    split_dict = {"train": train_list, "val": val_list}

    output_path = os.path.join(OUTPUT_DIR, "dataset_split.json")

    try:
        with open(output_path, "w") as f:
            json.dump(split_dict, f, indent=4)
        print(f"✅ Split saved to: {output_path}")
    except OSError as e:
        print(f"❌ Failed to save JSON. Error: {e}")
        print("Manual Split Data:", split_dict)

    print(f" - Training: {len(train_list)}, Validation: {len(val_list)}")
    return split_dict


def load_dataset_split():
    path = os.path.join(OUTPUT_DIR, "dataset_split.json")
    with open(path, "r") as f:
        return json.load(f)


# FIX: Check if split exists before recreating
split_path = os.path.join(OUTPUT_DIR, "dataset_split.json")
if os.path.exists(split_path):
    print(f"📄 Loading existing split from {split_path}")
    dataset_split = load_dataset_split()
    print(f"   {len(dataset_split['train'])} train / {len(dataset_split['val'])} val")
else:
    print("🔀 Creating train/val split...")
    dataset_split = save_dataset_split(all_cases)

# FIX: Verify no leakage
overlap = set(dataset_split['train']) & set(dataset_split['val'])
if overlap:
    print(f"❌ LEAKAGE: {len(overlap)} cases in both partitions!")
else:
    print(f"✅ No leakage: {len(set(dataset_split['train']))} train / "
          f"{len(set(dataset_split['val']))} val, 0 overlap")


📄 Loading existing split from /content/drive/MyDrive/Group_Project_766/dataset_split.json
   1000 train / 251 val
✅ No leakage: 1000 train / 251 val, 0 overlap


In [7]:
# ═══════════════════════════════════════════════════════════════
# SEG-04: Partition-Aware 2D Axial Slice Extraction
#
# FIX: +1 on end_z to include full margin in range().
# FIX: Partition-aware — train and val saved to separate dirs.
# ═══════════════════════════════════════════════════════════════

def extract_tumor_slices(data_dict, margin=5):
    seg = data_dict['seg']
    z_indices = np.any(seg > 0, axis=(0, 1))

    if not np.any(z_indices):
        return [], []

    start_z = max(0, np.where(z_indices)[0].min() - margin)
    # FIX: + 1 so range() includes the last margin slice
    end_z = min(seg.shape[2], np.where(z_indices)[0].max() + margin + 1)

    slices_images, slices_masks = [], []
    for z in range(start_z, end_z):
        combined_slice = np.stack([
            data_dict['t1'][:, :, z],
            data_dict['t1ce'][:, :, z],
            data_dict['t2'][:, :, z],
            data_dict['flair'][:, :, z]
        ], axis=0)
        slices_images.append(combined_slice)
        slices_masks.append(seg[:, :, z])
    return slices_images, slices_masks


def extract_partition(partition_case_ids, partition_name):
    """Extract slices for one partition to slices/{train|val}/.
    Calls load + normalize inside the loop."""
    save_dir = os.path.join(OUTPUT_DIR, "slices", partition_name)
    os.makedirs(save_dir, exist_ok=True)

    total = 0
    for i, case_id in enumerate(partition_case_ids):
        data_dict = load_case_data(case_id)
        data_dict = normalize_case(data_dict)
        imgs, msks = extract_tumor_slices(data_dict)

        for j, (im, mk) in enumerate(zip(imgs, msks)):
            np.savez_compressed(
                os.path.join(save_dir, f"{case_id}_slice_{j:03d}.npz"),
                image=im, mask=mk)

        total += len(imgs)
        if (i + 1) % 50 == 0 or (i + 1) == len(partition_case_ids):
            print(f"  [{partition_name}] {i+1}/{len(partition_case_ids)}, "
                  f"{total} slices")

    print(f"✅ {partition_name}: {total} slices → {save_dir}")
    return total


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Run Partition-Aware Extraction
# This will take a while for all 1,251 cases.
# ═══════════════════════════════════════════════════════════════

print("📤 Extracting training slices...")
train_count = extract_partition(dataset_split['train'], 'train')

print("\n📤 Extracting validation slices...")
val_count = extract_partition(dataset_split['val'], 'val')

print(f"\n✅ Total: {train_count} train + {val_count} val slices")


📤 Extracting training slices...
  [train] 50/1000, 3744 slices
  [train] 100/1000, 7571 slices
  [train] 150/1000, 11570 slices
  [train] 200/1000, 15572 slices
  [train] 250/1000, 19530 slices
  [train] 300/1000, 23263 slices
  [train] 350/1000, 27012 slices
  [train] 400/1000, 30840 slices
  [train] 450/1000, 34733 slices
  [train] 500/1000, 38454 slices
  [train] 550/1000, 41970 slices
  [train] 600/1000, 45753 slices
  [train] 650/1000, 49357 slices
  [train] 700/1000, 53205 slices
  [train] 750/1000, 56946 slices
  [train] 800/1000, 60778 slices
  [train] 850/1000, 64749 slices
  [train] 900/1000, 68557 slices


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-05: Train-Only Augmentation
#
# FIX: .copy() after np.flip
# FIX: Geometric (image+mask) separated from photometric (image only)
# FIX: borderMode=BORDER_CONSTANT, borderValue=0.0
# FIX: Mask cast float32 before warp, uint8 after
# FIX: Wrapped in Dataset class with augment flag
# ═══════════════════════════════════════════════════════════════

def geometric_augmentation(image, mask):
    """GEOMETRIC: apply to BOTH image and mask. NN interp on mask."""
    if random.random() > 0.5:
        image = np.flip(image, axis=2).copy()  # FIX: .copy()
        mask = np.flip(mask, axis=1).copy()     # FIX: .copy()

    angle = random.uniform(-15, 15)
    h, w = mask.shape
    rot = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)

    aug_img = np.zeros_like(image)
    for c in range(image.shape[0]):
        aug_img[c] = cv2.warpAffine(
            image[c], rot, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)

    aug_msk = cv2.warpAffine(
        mask.astype(np.float32), rot, (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0.0
    ).astype(np.uint8)

    return aug_img, aug_msk


def photometric_augmentation(image):
    """PHOTOMETRIC: IMAGE ONLY. Never apply to mask."""
    image = image * random.uniform(0.9, 1.1)
    if random.random() > 0.5:
        noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
        image = image + noise
    return image


def apply_augmentation(image, mask):
    """Geometric first (both), then photometric (image only)."""
    image, mask = geometric_augmentation(image, mask)
    image = photometric_augmentation(image)
    return image, mask


class BraTSSliceDataset:
    """Dataset wrapper. augment=True for train, False for val."""
    def __init__(self, slice_dir, augment=False):
        self.slice_dir = slice_dir
        self.augment = augment
        self.file_list = sorted(
            [f for f in os.listdir(slice_dir) if f.endswith('.npz')])
        print(f"📦 {len(self.file_list)} slices "
              f"(augment={'ON' if augment else 'OFF'})")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        data = np.load(os.path.join(self.slice_dir, self.file_list[idx]))
        image = data['image'].astype(np.float32)
        mask = data['mask'].astype(np.uint8)

        if self.augment:
            image, mask = apply_augmentation(image, mask)

        return image.astype(np.float32), mask.astype(np.int64)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Create Datasets and Verify
# ═══════════════════════════════════════════════════════════════

train_dir = os.path.join(OUTPUT_DIR, 'slices', 'train')
val_dir = os.path.join(OUTPUT_DIR, 'slices', 'val')

train_dataset = BraTSSliceDataset(train_dir, augment=True)
val_dataset = BraTSSliceDataset(val_dir, augment=False)

# Demo: load one sample from each
if len(train_dataset) > 0:
    img_t, msk_t = train_dataset[0]
    print(f"\n🔬 Train sample: image {img_t.shape} {img_t.dtype}, "
          f"mask {msk_t.shape} {msk_t.dtype}")
    print(f"   Mask labels: {np.unique(msk_t)}")

if len(val_dataset) > 0:
    img_v, msk_v = val_dataset[0]
    print(f"🔬 Val sample:   image {img_v.shape} {img_v.dtype}, "
          f"mask {msk_v.shape} {msk_v.dtype}")
    print(f"   Mask labels: {np.unique(msk_v)}")

print("\n🚀 Pipeline complete. Ready for model training.")

# For PyTorch DataLoader, uncomment:
# import torch
# from torch.utils.data import DataLoader
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
